# 00 Setup


In [3]:
import json, os
from pathlib import Path

ROOT    = Path(os.getcwd())
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
Q1_PATH = ROOT / "benchmark" / "questionset_v2_part1.json"
Q2_PATH = ROOT / "benchmark" / "questionset_v2_part2.json"

# SQLite 순수 산술만 사용하는 분산 버전 (EXP/LOG/SQRT 모두 불필요)
NEW_SQL = (
    "SELECT "
    "(i.age_at_entry / 10) * 10 AS age_group_start, "
    "(i.age_at_entry / 10) * 10 + 9 AS age_group_end, "
    "COUNT(ct.contract_id) AS contract_count, "
    "ROUND(AVG(ct.sum_insured), 0) AS avg_sum_insured, "
    "ROUND("
    "  AVG(ct.sum_insured * ct.sum_insured) - AVG(ct.sum_insured) * AVG(ct.sum_insured)"
    ", 0) AS variance_sum_insured "
    "FROM contracts ct "
    "JOIN insured i ON i.insured_id = ct.insured_id "
    "JOIN products pr ON ct.product_id = pr.product_id "
    "WHERE pr.insurance_type = 'life' "
    "AND ct.contract_status = 'active' "
    "GROUP BY age_group_start, age_group_end "
    "ORDER BY age_group_start"
)

# part1, part2 모두 탐색
found = False
for path in [Q1_PATH, Q2_PATH]:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    questions = data if isinstance(data, list) else data.get("questions", [])
    for q in questions:
        if q.get("question_id") == "UW068":
            q["gold_sql"] = NEW_SQL
            q["gold_sql_note"] = "SQLite 수학함수 미지원 환경: 표준편차 대신 분산(variance) 출력. 표준편차 = sqrt(variance)"
            found = True
            out = questions if isinstance(data, list) else {**data, "questions": questions}
            with open(path, "w", encoding="utf-8") as f:
                json.dump(out, f, ensure_ascii=False, indent=2)
            print(f"✅ UW068 수정 완료 → {path.name}")
            break
    if found:
        break

if not found:
    print("❌ UW068 를 찾을 수 없습니다")

# ── DB에서 직접 실행 테스트
DB_PATH = ROOT / "data" / "insurance_uw.db"
conn = sqlite3.connect(DB_PATH)
try:
    conn.execute(NEW_SQL)
    print("✅ SQLite 실행 성공")
except Exception as e:
    print(f"❌ 오류: {e}")
finally:
    conn.close()

print("\n→ 00_setup 셀 8 재실행하여 80/80 확인하세요")

✅ UW068 수정 완료 → questionset_v2_part2.json
✅ SQLite 실행 성공

→ 00_setup 셀 8 재실행하여 80/80 확인하세요


In [4]:
# ============================================================
# 00_setup.ipynb
# 환경 확인 → DB 생성 → 샘플 데이터 삽입 → 질문셋 로드 검증
# ============================================================
# 셀 단위로 구분 표시: # %% [셀 제목]
# Jupyter에서 각 셀을 순서대로 실행하세요.
# ============================================================

# %% [1] 라이브러리 임포트 & 경로 설정
import sqlite3
import json
import os
import sys
import random
import string
from pathlib import Path
from datetime import date, timedelta
import pandas as pd

# 프로젝트 루트 기준 경로
ROOT       = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
SCHEMA_SQL = ROOT / "schema" / "schema.sql"
DB_PATH    = ROOT / "data"   / "insurance_uw.db"
BM_DIR     = ROOT / "benchmark"
Q1_PATH    = BM_DIR / "questionset_v2_part1.json"
Q2_PATH    = BM_DIR / "questionset_v2_part2.json"

print("=" * 55)
print("  Insurance UW NL-to-SQL Benchmark  |  00_setup")
print("=" * 55)
print(f"ROOT       : {ROOT}")
print(f"SCHEMA_SQL : {SCHEMA_SQL}")
print(f"DB_PATH    : {DB_PATH}")
print(f"Python     : {sys.version}")
print(f"pandas     : {pd.__version__}")


# %% [2] 경로 존재 확인
assert SCHEMA_SQL.exists(), f"schema.sql 없음: {SCHEMA_SQL}"
assert Q1_PATH.exists(),    f"questionset_v2_part1.json 없음: {Q1_PATH}"
assert Q2_PATH.exists(),    f"questionset_v2_part2.json 없음: {Q2_PATH}"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
print("✅ 경로 확인 완료")


# %% [3] DB 생성 & 스키마 적용
def create_db(db_path: Path, schema_path: Path) -> sqlite3.Connection:
    if db_path.exists():
        db_path.unlink()
        print(f"  기존 DB 삭제: {db_path.name}")
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("PRAGMA journal_mode = WAL")
    ddl = schema_path.read_text(encoding="utf-8")
    conn.executescript(ddl)
    conn.commit()
    print(f"  ✅ DB 생성 완료: {db_path.name}")
    return conn

conn = create_db(DB_PATH, SCHEMA_SQL)

# 테이블 목록 확인
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
print(f"\n생성된 테이블 ({len(tables)}개):")
print(tables.to_string(index=False))


# %% [4] 헬퍼 함수
def rnd_id(prefix: str = "", n: int = 6) -> str:
    return prefix + "".join(random.choices(string.digits, k=n))

def date_offset(base: str, days: int = 0, years: int = 0) -> str:
    d = date.fromisoformat(base) + timedelta(days=days + years * 365)
    return d.isoformat()

def rand_date(start="2015-01-01", end="2024-12-31") -> str:
    s = date.fromisoformat(start)
    e = date.fromisoformat(end)
    return (s + timedelta(days=random.randint(0, (e - s).days))).isoformat()


# %% [5] 샘플 데이터 삽입
random.seed(42)

# ── 5-1. 보험회사
companies = [
    ("CO001", "한국생명보험", 142.3, 138.5),
    ("CO002", "미래생명보험", 89.7,  85.2),
    ("CO003", "현대생명보험", 210.1, 205.8),
]
conn.executemany(
    "INSERT INTO insurance_companies VALUES (?,?,?,?,DATE('now'))", companies
)

# ── 5-2. 보험상품
products = [
    # (product_id, name, category, type, group, insurance_type, med_subtype, contract_type,
    #  period, period_yr, pay_yr, pay_type, pay_cycle, ref_age,
    #  surv_total, exp_prem, min_guar, inp_ded, last_cov, auto_ren, low_surr, post_annuity,
    #  concl_cost, std_surr, cost_idx, front_ratio, dist_type)
    ("PR001","종신보험A","protection","whole_life","individual","life",None,"main",
     "whole_life",None,20,"full_term","monthly",None,
     0,24000000,None,None,"2020-01-01",0,0,0,
     1200000,800000,1.05,0.3,"uniform"),
    ("PR002","저축보험B","savings","savings","individual","life",None,"main",
     "10",10,10,"full_term","monthly",40,
     15000000,12000000,None,None,"2019-06-01",0,0,0,
     600000,500000,1.02,0.6,"front_loaded"),
    ("PR003","실손의료C","third_insurance","actual_loss_medical","individual","life","basic","main",
     "1",1,1,"full_term","monthly",None,
     0,600000,None,0.20,"2022-03-01",1,0,0,
     30000,25000,1.01,0.1,"uniform"),
    ("PR004","연금보험D","annuity","annuity","individual","life",None,"main",
     "whole_life",None,20,"full_term","monthly",None,
     20000000,18000000,None,None,"2018-09-01",0,0,1,
     900000,700000,1.03,0.2,"uniform"),
    ("PR005","변액보험E","variable","variable","individual","life",None,"main",
     "whole_life",None,20,"full_term","monthly",None,
     0,24000000,None,None,"2021-01-01",0,0,0,
     1200000,900000,1.04,0.25,"uniform"),
    ("PR006","제3보험F","third_insurance","third_insurance","individual","life",None,"main",
     "20",20,20,"full_term","monthly",None,
     0,12000000,None,None,"2017-01-01",0,0,0,
     600000,400000,1.02,0.15,"uniform"),
    ("PR007","금리연동G","savings","interest_linked","individual","life",None,"main",
     "10",10,10,"full_term","monthly",None,
     12000000,10000000,0.02,None,"2020-05-01",0,0,0,
     500000,420000,1.01,0.1,"uniform"),
    ("PR008","외화보험H","protection","foreign_currency","individual","life",None,"main",
     "whole_life",None,20,"full_term","monthly",None,
     0,18000000,None,None,"2021-06-01",0,0,0,
     900000,700000,1.03,0.2,"uniform"),
]
conn.executemany("""
    INSERT INTO products VALUES (
        ?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,DATE('now')
    )""", products)

# ── 5-3. 모집종사자
agents_data = [
    ("AG001","김철수","한국대리점"),
    ("AG002","이영희","미래대리점"),
    ("AG003","박민준","현대대리점"),
    ("AG004","최지은","한국대리점"),
]
conn.executemany(
    "INSERT INTO agents(agent_id,agent_name,agency_name,created_at) VALUES(?,?,?,DATE('now'))",
    agents_data
)
conn.executemany(
    "INSERT INTO commission_plans VALUES(?,?)",
    [("CP001","split"),("CP002","standard")]
)

# ── 5-4. 피보험자
insured_rows = []
for i in range(1, 201):
    iid    = f"IN{i:04d}"
    birth  = rand_date("1950-01-01","2000-12-31")
    age    = 2024 - int(birth[:4])
    gender = random.choice(["M","F"])
    death  = rand_date("2020-01-01","2024-12-31") if i <= 20 else None
    insured_rows.append((iid, f"피보험자{i}", birth, gender, age, age, death, "2024-01-01"))
conn.executemany(
    "INSERT INTO insured VALUES (?,?,?,?,?,?,?,?)", insured_rows
)

# ── 5-5. 보험계약 (200건)
product_pool = ["PR001","PR002","PR003","PR004","PR005","PR006","PR007","PR008"]
status_pool  = ["active"]*140 + ["terminated"]*40 + ["expired"]*10 + ["withdrawn"]*10
term_reasons = [None]*140 + ["고지의무위반"]*10 + ["중대사유"]*8 + ["납입연체"]*12 + ["강제집행"]*5 + ["사기"]*5 + [None]*20
fraud_types  = [None]*168 + ["고의보험사고유발"]*8 + ["서류위변조"]*8 + ["principal_guarantee_solicitation"]*8 + [None]*8

contracts_rows = []
for i in range(1, 201):
    cid     = f"CT{i:04d}"
    pid     = random.choice(product_pool)
    iid     = f"IN{i:04d}"
    co      = random.choice(["CO001","CO002","CO003"])
    ag      = random.choice(["AG001","AG002","AG003","AG004"])
    cdate   = rand_date("2015-01-01","2023-12-31")
    cstart  = date_offset(cdate, days=1)
    status  = status_pool[i-1]
    tdate   = rand_date("2020-01-01","2024-06-30") if status == "terminated" else None
    treason = term_reasons[i-1]
    creason = random.choice(["대리진단","진단서위변조","사기",None]) if i <= 15 else None
    mprem   = random.choice([50000,100000,150000,200000,300000])
    aprem   = mprem * 12
    tprem   = mprem * random.randint(12, 120)
    sumin   = random.choice([10000000,30000000,50000000,100000000,200000000])
    dbenefit= sumin
    surr    = tprem * random.uniform(0.3, 0.9)
    surr_std= surr * random.uniform(1.1, 1.5)
    reserve = surr * random.uniform(0.7, 1.1)
    ph_res  = reserve * random.uniform(0.9, 1.1)
    loan    = surr * random.uniform(0, 0.5) if random.random() < 0.3 else 0
    pcnt    = random.randint(0, 120)
    pyr     = pcnt // 12
    age_c   = random.randint(20, 65)
    ins_age = age_c
    ctype   = random.choice(["진단계약","무진단계약"])
    hec     = 0 if ctype == "진단계약" and random.random() < 0.05 else 1
    ftype   = fraud_types[i-1]
    causation = 1 if treason == "고지의무위반" and random.random() < 0.6 else 0
    age_corr  = 1 if random.random() < 0.05 else 0
    orig_p    = mprem if age_corr else None
    corr_p    = mprem * 1.1 if age_corr else None
    annuity_s = date_offset(cdate, years=20) if pid == "PR004" and status == "active" else None
    curr      = "USD" if pid == "PR008" else "KRW"
    curr_prem = mprem * 1350 if curr == "USD" else mprem

    contracts_rows.append((
        # 1~5
        cid, pid, iid, co, ag,
        # 6~10
        f"PH{i:04d}", cdate, cstart, tdate, annuity_s,
        # 11~15
        status, treason, creason, "사망", ctype,
        # 16~20
        mprem, aprem, tprem, sumin, dbenefit,
        # 21~25
        None,          # min_death_benefit
        surr, surr_std, reserve, ph_res,
        # 26~30
        None,          # reserve_amount_at_7yr
        None,          # expected_total_premium_7yr
        loan, pcnt, pyr,
        # 31~35
        age_c, ins_age,
        0,             # third_party_death_coverage
        1,             # insured_written_consent
        hec,           # health_exam_completed
        # 36~40
        age_corr, orig_p, corr_p,
        0,             # disability_rate_revised
        ftype,
        # 41~44
        causation, curr, curr_prem,
        None,          # third_insurance_period_years
        # created_at → DATE('now') via SQL
    ))

conn.executemany("""
    INSERT INTO contracts (
        contract_id, product_id, insured_id, company_id, agent_id,
        policyholder_id, contract_date, coverage_start_date, termination_date, annuity_start_date,
        contract_status, termination_reason, cancellation_reason, coverage_type, contract_type,
        monthly_premium, annual_premium, total_paid_premium, sum_insured, death_benefit,
        min_death_benefit, surrender_value, surrender_value_standard_type,
        remaining_coverage_reserve, policyholder_reserve,
        reserve_amount_at_7yr, expected_total_premium_7yr,
        loan_balance, payment_count, payment_year_elapsed,
        insured_age_at_contract, insurance_age,
        third_party_death_coverage, insured_written_consent, health_exam_completed,
        age_correction_applied, original_premium, corrected_premium,
        disability_rate_revised, fraud_type,
        causation_proven, currency_code, current_monthly_premium_krw,
        third_insurance_period_years
    ) VALUES (
        ?,?,?,?,?,?,?,?,?,?,
        ?,?,?,?,?,?,?,?,?,?,
        ?,?,?,?,?,?,?,?,?,?,
        ?,?,?,?,?,?,?,?,?,?,
        ?,?,?,?
    )""", contracts_rows)

# ── 5-6. 언더라이팅 심사결과
uw_rows = []
for i, row in enumerate(contracts_rows, 1):
    cid = row[0]
    dtype = random.choices(
        ["승낙","조건부승낙","거절"], weights=[70,20,10]
    )[0]
    ctype2 = random.choice(["보험료할증","보장제외","보험금삭감"]) if dtype == "조건부승낙" else None
    surcharge = round(random.uniform(0.05, 0.50), 2) if dtype == "조건부승낙" else 0
    extra = round(random.uniform(0.10, 0.40), 2) if random.random() < 0.1 else 0
    uw_rows.append((
        f"UD{i:04d}", cid, row[1], dtype, ctype2,
        surcharge, surcharge, extra,
        random.randint(50000, 300000), row[6]
    ))
conn.executemany("""
    INSERT INTO underwriting_decisions VALUES (?,?,?,?,?,?,?,?,?,?,DATE('now'))
""", uw_rows)

# ── 5-7. 보험금 청구 (100건)
claim_types  = ["사망보험금","장해보험금","입원보험금","만기보험금"]
claim_status = ["지급완료","지급완료","지급완료","지급거절","심사중","소멸시효완성"]
dis_types    = ["심한추간판탈출증","흉복부장기심한장해","심한간질발작","치매","눈장해",None]
dis_body     = ["눈","귀","척추","팔","다리",None]

claims_rows = []
for i in range(1, 101):
    clid   = f"CL{i:04d}"
    cid    = f"CT{random.randint(1,200):04d}"
    iid    = f"IN{random.randint(1,200):04d}"
    ctype3 = random.choice(claim_types)
    cdate2 = rand_date("2020-01-01","2024-06-30")
    cstatus= random.choice(claim_status)
    paid   = random.randint(1000000, 100000000) if cstatus == "지급완료" else 0
    due    = date_offset(cdate2, days=3)
    actual = date_offset(cdate2, days=random.randint(1,60)) if cstatus == "지급완료" else None
    bdays  = random.randint(1,30) if cstatus == "지급완료" else None
    add_int= paid * 0.04 * random.randint(0,30) / 365 if bdays and bdays > 30 else 0
    denial = random.choice(["수익자고의피해","재해분류제외",None]) if cstatus == "지급거절" else None
    dcause = random.choice(["자살","질병","재해",None]) if ctype3 == "사망보험금" else None
    dctype = random.choice(["실종선고",None]) if dcause else None
    icd    = random.choice(["W75","W76","W80",None])
    dis_t  = random.choice(dis_types) if ctype3 == "장해보험금" else None
    dis_b  = random.choice(dis_body) if ctype3 == "장해보험금" else None
    dis_rate = random.choice([10,20,30,50,60,75,100]) if ctype3 == "장해보험금" else None
    cdr    = random.randint(2,5) if dis_t == "치매" else None
    fix_days = random.randint(60, 300) if ctype3 == "장해보험금" else None
    pdiag  = random.choice(["뇌졸중","뇌손상",None])
    asses  = date_offset(cdate2, days=random.randint(30,300)) if pdiag else None
    onset  = date_offset(cdate2, days=-random.randint(1,60)) if pdiag else None
    prov   = 1 if random.random() < 0.1 else 0
    prov_amt = paid * 0.5 if prov else 0
    consent= 0 if random.random() < 0.05 else 1
    ins_evt= date_offset(cdate2, days=-random.randint(0,1000))
    expiry = date_offset(ins_evt, years=3)
    miss   = random.randint(365,1825) if dctype == "실종선고" else None
    pay_meth = random.choice(["일시지급","분할지급"])
    int_add = paid * 0.01 if pay_meth == "분할지급" else 0

    claims_rows.append((
        clid, cid, iid, ctype3, cdate2, cstatus,
        paid, due, actual, add_int, bdays, pay_meth, int_add,
        prov, prov_amt, consent, denial, None, dcause, dctype,
        icd, miss, dis_t, None, dis_b, dis_rate, fix_days,
        cdr, pdiag, asses, onset, None, ins_evt, expiry, "2024-01-01"
    ))

conn.executemany("""
    INSERT INTO claims VALUES (
        ?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,
        ?,?,?,?,?,?,?,?,?,?,?,?,?,?,?
    )""", claims_rows)

# ── 5-8. 건강이력
mh_rows = []
for i in range(1, 51):
    iid   = f"IN{random.randint(1,200):04d}"
    cid   = f"CT{random.randint(1,200):04d}"
    diag  = random.choice(["E11","I10","C34","J45","M54"])
    prior = diag
    tdate2 = rand_date("2010-01-01","2023-12-31")
    cdate3 = date_offset(tdate2, years=-2)
    mh_rows.append((f"MH{i:04d}", iid, cid, diag, prior, tdate2, cdate3, "2024-01-01"))
conn.executemany("INSERT INTO medical_history VALUES (?,?,?,?,?,?,?,?)", mh_rows)

# ── 5-9. 보장제외
ex_rows = [(f"EX{i:04d}", f"CT{random.randint(1,200):04d}",
            random.choice(["암","당뇨","고혈압","정신질환"]), "2024-01-01")
           for i in range(1, 31)]
conn.executemany("INSERT INTO contract_exclusions VALUES (?,?,?,?)", ex_rows)

# ── 5-10. 계약부활
rev_rows = []
for i in range(1, 21):
    cid    = f"CT{random.randint(141,180):04d}"
    tdate3 = rand_date("2020-01-01","2022-12-31")
    rdate  = date_offset(tdate3, days=random.randint(30, 900))
    rev_rows.append((
        f"RV{i:04d}", cid, random.choice(product_pool), None,
        random.choice(["부활","특별부활"]),
        random.choice(["납입연체","강제집행","담보권실행","체납처분"]),
        rdate, tdate3,
        rand_date("2023-01-01","2024-12-31"),
        random.randint(50000,200000), random.randint(50000,200000),
        random.randint(500000,2000000), random.randint(500000,2000000),
        random.randint(0,1), random.randint(0,1), "2024-01-01"
    ))
conn.executemany("""
    INSERT INTO contract_revivals VALUES (
        ?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?
    )""", rev_rows)

# ── 5-11. 청약철회
wd_rows = [(f"WD{i:04d}", f"CT{random.randint(181,200):04d}",
            rand_date("2023-01-01","2024-06-30"),
            rand_date("2023-01-01","2023-06-30"), "2024-01-01")
           for i in range(1, 16)]
conn.executemany("INSERT INTO contract_withdrawals VALUES (?,?,?,?,?)", wd_rows)

# ── 5-12. 계약변경
cc_rows = []
for i in range(1, 21):
    cid    = f"CT{random.randint(1,200):04d}"
    rdate  = rand_date("2022-01-01","2024-06-30")
    fpdate = date_offset(rdate, years=-random.randint(0,2))
    cc_rows.append((
        f"CC{i:04d}", cid, "보험종목변경",
        random.choice(["승인","거절"]), rdate, fpdate, "2024-01-01"
    ))
conn.executemany("INSERT INTO contract_changes VALUES (?,?,?,?,?,?,?)", cc_rows)

# ── 5-13. 납입최고
pn_rows = []
for i in range(1, 31):
    cid   = f"CT{random.randint(141,180):04d}"
    ndate = rand_date("2021-01-01","2024-01-01")
    paid_within = random.randint(0,1)
    apdate = date_offset(ndate, days=random.randint(1,14)) if paid_within else None
    pn_rows.append((f"PN{i:04d}", cid, ndate, apdate, paid_within, "2024-01-01"))
conn.executemany("INSERT INTO premium_notices VALUES (?,?,?,?,?,?)", pn_rows)

# ── 5-14. 대출차감
ld_rows = [(f"LD{i:04d}", f"CT{random.randint(1,200):04d}",
            "해지환급금차감",
            rand_date("2022-01-01","2024-06-30"),
            random.randint(100000,5000000), "2024-01-01")
           for i in range(1, 21)]
conn.executemany("INSERT INTO loan_deductions VALUES (?,?,?,?,?,?)", ld_rows)

# ── 5-15. 수수료
cp_rows = [(f"CP{i:04d}", f"CT{i:04d}", random.choice(["AG001","AG002","AG003","AG004"]),
            random.randint(50000,500000), rand_date("2015-01-01","2024-01-01"), "2024-01-01")
           for i in range(1, 51)]
conn.executemany("INSERT INTO commission_payments VALUES (?,?,?,?,?,?)", cp_rows)

# ── 5-16. 모집채널
sc_rows = [(f"SC{i:04d}", f"CT{i:04d}",
            random.choice(["telemarketing","bancassurance","agency","direct"]),
            random.choices([0,1], weights=[5,95])[0], "2024-01-01")
           for i in range(1, 201)]
conn.executemany("INSERT INTO sales_channels VALUES (?,?,?,?,?)", sc_rows)

# ── 5-17. 품질점검
qc_rows = [(f"QC{i:04d}", f"CT{i:04d}",
            rand_date("2023-01-01","2024-06-30"),
            random.choices(["pass","insufficient_explanation","fail"], weights=[80,15,5])[0],
            "2024-01-01")
           for i in range(1, 51)]
conn.executemany("INSERT INTO quality_checks VALUES (?,?,?,?,?)", qc_rows)

# ── 5-18. 위반이력
vr_rows = [(f"VR{i:04d}", random.choice(["AG001","AG002","AG003","AG004"]),
            random.choice(["principal_guarantee_solicitation","unfair_sale"]),
            rand_date("2022-01-01","2024-01-01"), "2024-01-01")
           for i in range(1, 11)]
conn.executemany("INSERT INTO violation_records VALUES (?,?,?,?,?)", vr_rows)

# ── 5-19. 판매통계
ass_rows = [(f"AS{i:04d}", f"AG{i:03d}",
             random.choice([2022,2023]),
             random.randint(50,200), random.randint(0,5), "2024-01-01")
            for i in range(1, 5)]
conn.executemany("INSERT INTO agent_sales_stats VALUES (?,?,?,?,?,?)", ass_rows)

# ── 5-20. 위험액
rk_rows = [(f"RK{i:04d}", random.choice(product_pool),
            random.choice([20241,20242,20243,20244]),
            random.randint(100000000,5000000000),
            random.randint(50000000,2000000000), "2024-01-01")
           for i in range(1, 33)]
conn.executemany("INSERT INTO risk_capitals VALUES (?,?,?,?,?,?)", rk_rows)

# ── 5-21. 자산건전성
alc_rows = [(f"AL{i:04d}", f"CT{random.randint(1,200):04d}",
             random.choice(["household","corporate"]),
             random.choices(["normal","precautionary","substandard","doubtful","loss"],
                            weights=[60,20,10,7,3])[0],
             random.randint(1000000,50000000),
             random.randint(0,5000000), "2024-01-01")
            for i in range(1, 31)]
conn.executemany("INSERT INTO asset_loan_classifications VALUES (?,?,?,?,?,?,?)", alc_rows)

# ── 5-22. 계리의견
ao_rows = [(f"AO{i:04d}", random.choice(product_pool),
            random.choice([2022,2023]),
            random.choice(["liability_reserve","risk_reserve"]),
            random.choices(["unqualified","adverse","qualified"], weights=[80,10,10])[0],
            "2024-01-01")
           for i in range(1, 11)]
conn.executemany("INSERT INTO actuary_opinions VALUES (?,?,?,?,?,?)", ao_rows)

# ── 5-23. 공시이율
dr_rows = [(f"DR{i:04d}", random.choice(product_pool),
            random.choice([2022,2023,2024]),
            random.choice(["type_a","type_b"]),
            round(random.uniform(0.025, 0.045), 4),
            round(random.uniform(0.027, 0.048), 4), "2024-01-01")
           for i in range(1, 21)]
conn.executemany("INSERT INTO declared_rate_info VALUES (?,?,?,?,?,?,?)", dr_rows)

# ── 5-24. 환율
currencies = ["USD","JPY","EUR","CNY"]
exr_rows = []
for i, curr in enumerate(currencies, 1):
    for yr in range(2020, 2025):
        for mo in [1,4,7,10]:
            exr_rows.append((
                f"ER{i:02d}{yr}{mo:02d}",
                curr,
                f"{yr}-{mo:02d}-01",
                round(random.uniform(1100,1400) if curr=="USD" else
                      random.uniform(8,12)     if curr=="JPY" else
                      random.uniform(1300,1500) if curr=="EUR" else
                      random.uniform(160,200),  4),
                "2024-01-01"
            ))
conn.executemany("INSERT OR IGNORE INTO exchange_rate_history VALUES (?,?,?,?,?)", exr_rows)

# ── 5-25. 계약묶음
cb_rows = [(f"CB{i:04d}", f"CT{random.randint(1,200):04d}", random.choice(product_pool), "2024-01-01")
           for i in range(1, 6)]
conn.executemany("INSERT INTO contract_bundles VALUES (?,?,?,?)", cb_rows)

conn.commit()
print("✅ 샘플 데이터 삽입 완료")


# %% [6] 데이터 건수 확인
print("\n[테이블별 데이터 건수]")
tables2 = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
rows_info = []
for t in tables2["name"]:
    cnt = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    rows_info.append({"table": t, "rows": cnt})
df_cnt = pd.DataFrame(rows_info)
print(df_cnt.to_string(index=False))


# %% [7] 질문셋 로드 & 검증
def load_questionset(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return data if isinstance(data, list) else data.get("questions", [])

qs1 = load_questionset(Q1_PATH)
qs2 = load_questionset(Q2_PATH)
all_qs = qs1 + qs2

print(f"\n질문셋 로드: {len(qs1)} + {len(qs2)} = {len(all_qs)}개")

# 카테고리·난이도·민감정보 분포
df_qs = pd.DataFrame(all_qs)
print("\n[카테고리 분포]")
print(df_qs["category"].value_counts().to_string())
print("\n[난이도 분포]")
print(df_qs["difficulty"].value_counts().to_string())
print("\n[민감정보 수준 분포]")
print(df_qs["sensitive_info_level"].value_counts().to_string())


# %% [8] gold_sql 간이 실행 테스트 (syntax 확인)
print("\n[gold_sql 간이 실행 테스트]")
ok, fail = 0, []

for q in all_qs:
    qid = q["question_id"]
    sql = q.get("gold_sql", "")
    try:
        conn.execute(sql)
        ok += 1
    except Exception as e:
        fail.append({"question_id": qid, "error": str(e)})

print(f"  ✅ 성공: {ok}/{len(all_qs)}")
if fail:
    print(f"  ❌ 실패: {len(fail)}개")
    df_fail = pd.DataFrame(fail)
    print(df_fail.to_string(index=False))
else:
    print("  🎉 전체 gold_sql 실행 성공!")


# %% [9] 마무리
conn.close()
print(f"\n{'='*55}")
print("  00_setup 완료")
print(f"  DB 위치: {DB_PATH}")
print(f"  다음: 01_gold_sql_validation.ipynb 실행")
print(f"{'='*55}")

  Insurance UW NL-to-SQL Benchmark  |  00_setup
ROOT       : C:\Users\miy\Downloads\1000_논문투고\1_양문일_교신저자\23_언더라이팅_Text_to_SQL\insurance-underwriting-nl2sql-benchmark
SCHEMA_SQL : C:\Users\miy\Downloads\1000_논문투고\1_양문일_교신저자\23_언더라이팅_Text_to_SQL\insurance-underwriting-nl2sql-benchmark\schema\schema.sql
DB_PATH    : C:\Users\miy\Downloads\1000_논문투고\1_양문일_교신저자\23_언더라이팅_Text_to_SQL\insurance-underwriting-nl2sql-benchmark\data\insurance_uw.db
Python     : 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
pandas     : 3.0.3
✅ 경로 확인 완료
  기존 DB 삭제: insurance_uw.db
  ✅ DB 생성 완료: insurance_uw.db

생성된 테이블 (26개):
                      name
          actuary_opinions
         agent_sales_stats
                    agents
asset_loan_classifications
                    claims
       commission_payments
          commission_plans
          contract_bundles
          contract_changes
       contract_exclusions
         contract_revivals
      contract_withdr